# World Foresight Framework — Data Extract & Transform

This notebook extracts raw data from source files in `/Raw Data/`, applies proxy-specific
transform functions and loads the results into `Final Data.xlsx` → `Timeseries Data` sheet.

**Each proxy has its own transform function** (`d{n}_etl.py`) that handles cleaning,
normalization, and reshaping into the standard long format. Default coverage is the
**last 10 years (2015–2025)**.

| Column | Description |
|--------|-------------|
| `id` | Proxy dimension ID: `D{n}` e.g. `D1` (links to `Proxy Information`) |
| `proxy_id` | Format: `D{n}_{ISO3}` e.g. `D1_USA` |
| `market` | Country ISO3 code e.g. `USA` |
| `year` | Year (integer) |
| `value` | Transformed, normalized value |
| `labels` | Display unit for UI e.g. `% of GDP` |
| `metric` | Value scale: `NONE`, `THOUSANDS`, `MILLIONS`, `BILLIONS` |

## Imports


In [ ]:
# Always needed if want to use the local version
%reload_ext autoreload
%autoreload 2

%cd /Users/dunglai/Documents/Việt Dũng/Personal Projects/World Foresight Framework/Data Preparation


In [ ]:
import openpyxl
import pandas as pd


## Shared Loader

Reusable function to append a transformed DataFrame into the `Timeseries Data` sheet.

In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

OUTPUT = "/Users/dunglai/Documents/Việt Dũng/Personal Projects/World Foresight Framework/Final Data.xlsx"
RAW_DIR = "/Users/dunglai/Documents/Việt Dũng/Personal Projects/World Foresight Framework/Raw Data/"

SHEET   = "Historical Data"
HEADERS = ["id", "proxy_id", "market", "year", "value", "labels", "metric"]


def load_to_excel(df: pd.DataFrame, output_file: str = OUTPUT) -> None:
    """
    Append a transformed proxy DataFrame to the Timeseries Data sheet.
    Expects df with columns: proxy_id, market, year, value, labels, metric.
    The `id` column is the proxy dimension (e.g. 'D1'), derived from proxy_id.
    Skips rows already present (deduplication by proxy_id + year).
    """
    wb = openpyxl.load_workbook(output_file)
    ws = wb[SHEET]

    thin      = Side(style="thin", color="D0D0D0")
    border    = Border(left=thin, right=thin, top=thin, bottom=thin)
    dat_font  = Font(name="Arial", size=10)
    ctr_align = Alignment(horizontal="center", vertical="center")

    # Existing (proxy_id, year) keys for dedup; row_count drives the zebra striping
    last_row      = ws.max_row
    existing_keys = set()
    row_count     = 0
    if last_row > 1:
        for r in range(2, last_row + 1):
            pid  = ws.cell(row=r, column=2).value
            year = ws.cell(row=r, column=4).value
            if pid:
                existing_keys.add((pid, year))
                row_count += 1

    written = skipped = 0
    for _, row in df.iterrows():
        key = (row["proxy_id"], row["year"])
        if key in existing_keys:
            skipped += 1
            continue

        last_row  += 1
        row_count += 1
        proxy_dim = str(row["proxy_id"]).split("_")[0]          # 'D1_USA' -> 'D1'
        alt = PatternFill("solid", start_color="F2F7FC") if row_count % 2 == 0 else None
        vals = [proxy_dim, row["proxy_id"], row["market"],
                row["year"], row["value"], row["labels"], row["metric"]]

        for col, v in enumerate(vals, 1):
            c = ws.cell(row=last_row, column=col, value=v)
            c.font = dat_font; c.alignment = ctr_align; c.border = border
            if alt: c.fill = alt
        written += 1

    wb.save(output_file)
    print(f"  Written : {written} rows")
    if skipped:
        print(f"  Skipped : {skipped} rows (already exist)")

## Coverage Check

Quick QA to run right after a transform — `check_coverage(df)` reports datapoints per country, how many of the 35-country universe are covered, and which are **missing** (ISO3 + name).

In [ ]:
# Canonical 34-country universe: ISO3 -> display name (single source of truth)
MARKETS_34 = {
    "USA": "United States",      "CAN": "Canada",          "MEX": "Mexico",
    "BRA": "Brazil",             "ARG": "Argentina",       "DEU": "Germany",
    "FRA": "France",             "GBR": "United Kingdom",  "ITA": "Italy",
    "RUS": "Russia",             "TUR": "Turkey",          "POL": "Poland",
    "NLD": "Netherlands",        "UKR": "Ukraine",         "CHN": "China",
    "JPN": "Japan",              "KOR": "South Korea",     "IDN": "Indonesia",
    "AUS": "Australia",          "VNM": "Vietnam",         "KAZ": "Kazakhstan",
    "IND": "India",              "PAK": "Pakistan",        "BGD": "Bangladesh",
    "SAU": "Saudi Arabia",       "ARE": "United Arab Emirates", "IRN": "Iran",
    "ISR": "Israel",             "EGY": "Egypt",           "NGA": "Nigeria",
    "ZAF": "South Africa",       "ETH": "Ethiopia",        "KEN": "Kenya",
    "COD": "DR Congo"       
}


def check_coverage(df: pd.DataFrame, market_col: str = "market",
                   year_col: str = "year", universe: dict = None,
                   verbose: bool = True) -> pd.DataFrame:
    """
    Quick coverage QA for a transformed long-format DataFrame.

    Returns
    -------
    pd.DataFrame [market, country, datapoints], one row per country present.
    """
    universe = universe or MARKETS_34
    expected = set(universe)
    present  = set(df[market_col].unique()) if len(df) else set()

    counts = (df.groupby(market_col).size().reset_index(name="datapoints")
              if len(df) else pd.DataFrame(columns=[market_col, "datapoints"]))
    counts["country"] = counts[market_col].map(lambda m: universe.get(m, "??? (not in universe)"))
    counts = counts[[market_col, "country", "datapoints"]].sort_values(
        ["datapoints", market_col], ascending=[False, True]).reset_index(drop=True)

    missing = sorted(expected - present)
    extra   = sorted(present - expected)

    if verbose:
        proxy = (df["proxy_id"].iloc[0].split("_")[0]
                 if "proxy_id" in df.columns and len(df) else "?")
        yrs = (f"{int(df[year_col].min())}–{int(df[year_col].max())}"
               if year_col in df.columns and len(df) else "n/a")
        print(f"=== Coverage check [{proxy}] ===")
        print(f"Countries with data : {len(present & expected)} / {len(expected)}")
        print(f"Total datapoints    : {len(df)}    |    years: {yrs}")
        if len(counts):
            print(f"Datapoints/country  : min {counts.datapoints.min()}, "
                  f"max {counts.datapoints.max()}")
        print()
        if missing:
            print(f"Missing ({len(missing)}):")
            for iso in missing:
                print(f"   {iso}   {universe[iso]}")
        else:
            print("Missing: none — full coverage ✅")
        if extra:
            print(f"\n⚠ Unexpected markets not in universe ({len(extra)}): {extra}")

    return counts


---
## D1 — Raw Military Budget
**Source:** SIPRI Military Expenditure Database — `Current US$` sheet  
**Transform:** Raw current USD military expenditure, reported in USD millions.


In [ ]:
from Transform_Functions.d1_etl import extract_transform

file_name = "SIPRI-Milex-data-1949-2025_v1.2.xlsx"
path = RAW_DIR + file_name
df_d1 = extract_transform(path)

df_d1.head(5)

In [ ]:
check_coverage(df_d1)

In [ ]:
load_to_excel(df_d1)
print("Done.")

---
## D2 - Raw Total Trade
**Source:** World Bank Data360 API - exports + imports of goods and services.
**Transform:** Current USD total trade, auto-scaled to one common metric.


In [ ]:
from Transform_Functions.d2_etl import extract_transform as d2_extract_transform

df_d2 = d2_extract_transform()
df_d2.head(5)


In [ ]:
check_coverage(df_d2)


In [ ]:
load_to_excel(df_d2)
print("Done.")


---
## D4 - Raw GDP
**Source:** World Bank Data360 API - `NY.GDP.MKTP.CD`.
**Transform:** Current USD GDP, auto-scaled to one common metric.


In [ ]:
from Transform_Functions.d4_etl import extract_transform as d4_extract_transform

df_d4 = d4_extract_transform()
df_d4.head(5)


In [ ]:
check_coverage(df_d4)


In [ ]:
load_to_excel(df_d4)
print("Done.")


---
## D3 — Global Soft Power Score
**Source:** Brand Finance Global Soft Power Index  

In [ ]:
path = RAW_DIR + "D3_soft_power_scores.csv"
df_d3 = pd.read_csv(path)
df_d3["metric"] = None
df_d3

In [ ]:
check_coverage(df_d3)

In [ ]:
load_to_excel(df_d3)
print("Done.")

---
## D13 — Raw Total Exports of Fuel, Metals & Food
**Source:** World Bank Open Data API  
**Transform:** Merchandise exports × (`fuel % + ores/metals % + food %`) in current USD, auto-scaled to one common metric.


In [ ]:
from Transform_Functions.d13_etl import extract_transform as d13_extract_transform

df_d13 = d13_extract_transform()
df_d13.head(5)


In [ ]:
check_coverage(df_d13)


In [ ]:
load_to_excel(df_d13)
print("Done.")


---
## D15 — Raw Outward FDI
**Source:** World Bank Open Data API — `BM.KLT.DINV.CD.WD`  
**Transform:** Foreign direct investment net outflows in current USD, positive rows only, auto-scaled to one common metric.


In [ ]:
from Transform_Functions.d15_etl import extract_transform as d15_extract_transform

df_d15 = d15_extract_transform()
df_d15.head(5)


In [ ]:
check_coverage(df_d15)


In [ ]:
load_to_excel(df_d15)
print("Done.")


---
## D16 — Raw R&D Spending
**Source:** World Bank Open Data API  
**Transform:** GDP × R&D spending % of GDP in current USD, auto-scaled to one common metric.


In [ ]:
from Transform_Functions.d16_etl import extract_transform as d16_extract_transform

df_d16 = d16_extract_transform(start_year=2000, end_year=2024)
df_d16.head(5)


In [ ]:
check_coverage(df_d16)


In [ ]:
load_to_excel(df_d16)
print("Done.")


---
## D5 — Composite National Power Index (CINC)
**Source:** Correlates of War — National Material Capabilities v7  

In [ ]:
from Transform_Functions.d5_etl import extract_transform

nmc_path = RAW_DIR + "NMCv7"
df_d5 = extract_transform(nmc_path)
df_d5.head(5)

In [ ]:
check_coverage(df_d5)

In [ ]:
# Review df_d5 and the coverage report before loading.
load_to_excel(df_d5)
print("Done.")

---
## D6 — Geopolitical Alignment Score
**Source:** Voeten UN Ideal Point Estimates (Harvard Dataverse)

Positive = Western-aligned; negative = non-Western / revisionist (clusters with Russia, China, Iran).

In [ ]:
from Transform_Functions.d6_etl import extract_transform as d6_extract_transform

path = RAW_DIR + "Idealpointestimates1946-2025.csv"
df_d6 = d6_extract_transform(path)
df_d6.head(5)

In [ ]:
check_coverage(df_d6)

In [ ]:
# Review df_d6 and the coverage report before loading.
load_to_excel(df_d6)
print("Done.")

---
## D7 — Raw Eastern Bloc Arms Imports
**Source:** SIPRI Arms Transfers Database  
**Transform:** Sum delivered SIPRI TIV from Russia, China, North Korea, Iran, and Belarus by country-year.


In [ ]:
from Transform_Functions.d7_etl import extract_transform as d7_extract_transform

arms_path = RAW_DIR + "trade-register.csv"
df_d7 = d7_extract_transform(arms_path)
df_d7.head(10)

In [ ]:
check_coverage(df_d7)

In [ ]:
# Review df_d7 and the coverage report before loading.
load_to_excel(df_d7)
print("Done.")

---
## D8 — UN Voting Bloc Agreement (vs USA / China / Russia / India)
**Source:** Voeten UN GA dyadic agreement scores 
**Transform:** Share of UN GA votes a country casts the same way as each major power

In [ ]:
from Transform_Functions.d8_etl import extract_transform as d8_extract_transform

path = RAW_DIR + "AgreementScores.csv"
df_d8 = d8_extract_transform(path)
df_d8.head(8)

In [ ]:
check_coverage(df_d8)

In [ ]:
# Review df_d8 and the coverage report before loading.
load_to_excel(df_d8)
print("Done.")

---
## D9 — Trade Bloc Concentration (share of trade with USA / China / Russia)
**Source:** UN Comtrade export


In [ ]:
from Transform_Functions.d9_etl import extract_transform as d9_extract_transform

path = RAW_DIR + "TradeData_7_6_2026_0_23_25.csv"
df_d9 = d9_extract_transform(path)
df_d9.head(9)

In [ ]:
check_coverage(df_d9)

In [ ]:
# Review df_d9 and the coverage report before loading.
load_to_excel(df_d9)
print("Done.")

---
## D11 — Multilateral Treaty Ratification Rate
**Source:** UN Treaty Collection + Wikipedia (NPT, Geneva AP-I) → `Raw Data/UN_Treaties/`  
**Transform:** Cumulative UN-member parties per treaty / 193 UN members × 100


In [ ]:
from Transform_Functions.d11_etl import extract_transform as d11_extract_transform

treaty_dir = RAW_DIR + "UN_Treaties/"
df_d11 = d11_extract_transform(treaty_dir)
df_d11.head(12)

In [ ]:
load_to_excel(df_d11)
print("Done.")

---
## D12 — UN Peacekeeping Contribution
**Source:** UN Peace & Security Data Hub — DPO-UCHISTORICAL.csv  
**Transform:** Annual average monthly deployed personnel across all active missions


In [ ]:
from Transform_Functions.d12_etl import extract_transform as d12_extract_transform

path = RAW_DIR + "DPO-UCHISTORICAL.csv"
df_d12 = d12_extract_transform(path)
df_d12.tail(30)

In [ ]:
check_coverage(df_d12)

In [ ]:
load_to_excel(df_d12)
print("Done.")

---
## D14 — UN Majority Vote Alignment Rate
**Source:** Voeten UN GA Agreement Scores + Ideal Point Estimates  
**Transform:** Mean pairwise voting agreement with all other UN member states per year × 100


In [ ]:
from Transform_Functions.d14_etl import extract_transform as d14_extract_transform

df_d14 = d14_extract_transform(
    RAW_DIR + "AgreementScores.csv",
    RAW_DIR + "Idealpointestimates1946-2025.csv",
)
df_d14.head(8)

In [ ]:
check_coverage(df_d14)

In [ ]:
load_to_excel(df_d14)
print("Done.")